# Usage Events Silver Pipeline (Databricks)

In [ ]:
import sys
import os

# Repo root (Databricks-friendly)
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

print(f"[INFO] Repo root added: {repo_root}")

In [ ]:
print("[INFO] Starting USAGE EVENTS Silver pipeline")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.types import StructType
from delta.tables import DeltaTable
from functools import reduce

spark = SparkSession.builder.getOrCreate()

dbutils.widgets.text("batch_id", "default")
batch_id = dbutils.widgets.get("batch_id")

# 1. Read Bronze usage_events

In [ ]:
bronze_path = "/Volumes/datalake_catalog/datalake_schema/bronze/usage_events"

df_bronze = (
    spark.read.format("delta").load(bronze_path)
    .withColumnRenamed("created_at", "created_at_event")
)

df_bronze.show(2, truncate=False)
df_bronze.printSchema()

# 2. Parse + Flatten JSON

In [ ]:
json_rdd = df_bronze.select("data").rdd.map(lambda r: r[0])
json_schema = spark.read.json(json_rdd).schema

df_parsed = (
    df_bronze
    .withColumn("parsed", F.from_json(F.col("data"), json_schema))
    .select("*", "parsed.*")
    .drop("parsed")
)

def flatten_struct(df):
    while True:
        struct_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StructType)]
        if not struct_cols:
            break

        cols = []
        for f in df.schema.fields:
            if isinstance(f.dataType, StructType):
                for nested in f.dataType.fields:
                    cols.append(F.col(f"{f.name}.{nested.name}").alias(f"{f.name}_{nested.name}"))
            else:
                cols.append(F.col(f.name))

        df = df.select(*cols)
    return df

df_flat = flatten_struct(df_parsed).drop("data", "id")

# 3. Type Casting

In [ ]:
df1 = (
    df_flat
    .withColumn("created_at_event", F.col("created_at_event").cast("timestamp"))
    .withColumn("ingest_time", F.col("ingest_time").cast("timestamp"))
    .withColumn("processed", F.col("processed").cast("boolean"))
    .withColumn("content_id", F.col("content_id").cast("bigint"))
    .withColumn("subscription_id", F.col("subscription_id").cast("bigint"))
    .withColumn("user_id", F.col("user_id").cast("bigint"))
)

# 4. Deduplication

In [ ]:
w = Window.partitionBy("event_id", "event_timestamp").orderBy(F.col("ingest_time").desc())

df_ranked = df1.withColumn("rn", F.row_number().over(w))

df2 = df_ranked.filter("rn = 1").drop("rn")
df2_quarantine = df_ranked.filter("rn > 1").drop("rn").withColumn("validation_error", F.lit("duplicate event"))

# 5. Referential Integrity Check

In [ ]:
users_path = "/Volumes/datalake_catalog/datalake_schema/silver/users"
subs_path = "/Volumes/datalake_catalog/datalake_schema/silver/subscriptions"

df_users = spark.read.format("delta").load(users_path).select("user_id")
df_subs = spark.read.format("delta").load(subs_path).select("subscription_id")

df_ref = (
    df2
    .join(df_users.withColumn("u", F.lit(1)), "user_id", "left")
    .withColumn("missing_user", F.col("u").isNull())
    .drop("u")
    .join(df_subs.withColumn("s", F.lit(1)), "subscription_id", "left")
    .withColumn("missing_sub", F.col("s").isNull())
    .drop("s")
)

df3 = df_ref.filter(~F.col("missing_user") & ~F.col("missing_sub"))
df3_quarantine = df_ref.filter(F.col("missing_user") | F.col("missing_sub"))

# 6. Write to Silver Delta

In [ ]:
silver_path = "/Volumes/datalake_catalog/datalake_schema/silver/usage_events"

w = Window.partitionBy("event_id", "event_timestamp").orderBy(F.col("ingest_time").desc())
df_final = df3.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

if DeltaTable.isDeltaTable(spark, silver_path):
    delta = DeltaTable.forPath(spark, silver_path)
    delta.alias("t").merge(
        df_final.alias("s"),
        "t.event_id = s.event_id"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("overwrite").save(silver_path)

print("[DONE] Usage events written to Silver")